In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import datetime

#### Downloading and Preprocessing the data

In [2]:
BUFFER_SIZE = 70_000
BATCH_SIZE = 128
NUM_EPOCHS = 20

In [3]:
mnist_dataset, mnist_info = tfds.load(name = 'mnist', with_info = True, as_supervised=True)

In [4]:
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

In [5]:
def scale(image,label):
    image = tf.cast(image,tf.float32)
    image /= 255.
    return image,label


In [6]:
train_and_validation_data = mnist_train.map(scale)
test_data = mnist_test.map(scale)

In [7]:
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples,tf.int64)
num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples,tf.int64)

In [8]:
train_and_validation_data = train_and_validation_data.shuffle(BUFFER_SIZE)
train_data = train_and_validation_data.skip(num_validation_samples)
validation_data = train_and_validation_data.take(num_validation_samples)

In [9]:
train_data = train_data.batch(BATCH_SIZE)

In [10]:
validation_data = validation_data.batch(num_validation_samples)
test_data = test_data.batch(num_test_samples)

#### Create the model and train it

In [11]:
model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(28,28,1)),
    tf.keras.layers.Conv2D(50,5,activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Conv2D(50,3,activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10)
])

In [12]:
model.summary(line_length=75)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                   ┃ Output Shape            ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                │ (None, 24, 24, 50)      │        1,300 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ max_pooling2d (MaxPooling2D)   │ (None, 12, 12, 50)      │            0 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ conv2d_1 (Conv2D)              │ (None, 10, 10, 50)      │       22,550 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ max_pooling2d_1 (MaxPooling2D) │ (None, 5, 5, 50)        │            0 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ flatten (Flatten)              │ (None, 1250)            │            0 │
├────────────────────────────────┼─────────────────────────┼──────────────┤
│ dense (Dense)                  │ (None, 10)              │       12,510 │
└────────────────────────────────┴─────────────────────────┴──────────────┘

 Total params: 36,360 (142.03 KB)

 Trainable params: 36,360 (142.03 KB)

 Non-trainable params: 0 (0.00 B)

First dimension in all the layers is None because all data is batched. Thus we don't pass a single image of size 28*28*1 but actually passes a batch of thousand images all at once. That's supposed to be the first dimension, however model does not know our batch size just yet as we have not trained it. So, it leaves this dimension as None.    
      
Trainable parameters are just weights of our network. The parameters that the model is trying to learn. Sometimes, however, you may need to multiply your input by a constant number, say 2.6 and you would want that to happen everytime. In other words, this parameter should not be changed during the learning process. These kinds of parameters are non-trainable.   

In [13]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam',loss=loss_fn,metrics=['accuracy'])

In [14]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    mode='auto',
    min_delta=0,
    patience=2,
    verbose=0,
    restore_best_weights=True
)
# Here these parameters define that program should stop the training process when the validation loss starts to increase for two subsequent 
# epochs because patience=2.

In [15]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
# Keras 3 writes the model graph as a "keras" (conceptual) summary that TensorBoard's
# GRAPHS plugin cannot parse -> "Malformed GraphDef". Turn it off and export a real
# op graph ourselves in the cell below.
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir, histogram_freq=1, write_graph=False
)

In [16]:
model.fit(
    train_data,
    epochs = NUM_EPOCHS,
    callbacks = [tensorboard_callback,early_stopping],
    validation_data=validation_data,
    verbose =2 #this specifies what to print out during the training process. verbose = 2 means it will print info only at the end 
    # of each epoch while verbose = 1 displays progress bar for every batch
)

Epoch 1/20


/Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/53-Convolutional-Neural-Networks-with-TensorFlow-in-Python/.venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1787157895.652232 1145557 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


422/422 - 8s - 18ms/step - accuracy: 0.9239 - loss: 0.2673 - val_accuracy: 0.9775 - val_loss: 0.0829
Epoch 2/20
422/422 - 7s - 17ms/step - accuracy: 0.9790 - loss: 0.0704 - val_accuracy: 0.9850 - val_loss: 0.0508
Epoch 3/20
422/422 - 8s - 19ms/step - accuracy: 0.9834 - loss: 0.0545 - val_accuracy: 0.9845 - val_loss: 0.0433
Epoch 4/20
422/422 - 8s - 18ms/step - accuracy: 0.9874 - loss: 0.0425 - val_accuracy: 0.9883 - val_loss: 0.0367
Epoch 5/20
422/422 - 7s - 18ms/step - accuracy: 0.9896 - loss: 0.0354 - val_accuracy: 0.9883 - val_loss: 0.0393
Epoch 6/20
422/422 - 8s - 19ms/step - accuracy: 0.9904 - loss: 0.0316 - val_accuracy: 0.9907 - val_loss: 0.0315
Epoch 7/20
422/422 - 8s - 18ms/step - accuracy: 0.9918 - loss: 0.0281 - val_accuracy: 0.9933 - val_loss: 0.0232
Epoch 8/20
422/422 - 8s - 18ms/step - accuracy: 0.9924 - loss: 0.0247 - val_accuracy: 0.9937 - val_loss: 0.0203
Epoch 9/20
422/422 - 8s - 18ms/step - accuracy: 0.9935 - loss: 0.0218 - val_accuracy: 0.9953 - val_loss: 0.0135
Epo

In [17]:
# Export the op graph so the GRAPHS tab works (Keras 3's own graph summary is unparseable)
graph_writer = tf.summary.create_file_writer(log_dir + "/train")
concrete_fn = tf.function(lambda x: model(x)).get_concrete_function(
    tf.TensorSpec([1, 28, 28, 1], tf.float32)
)
with graph_writer.as_default():
    tf.summary.graph(concrete_fn.graph)
graph_writer.flush()
graph_writer.close()
print("graph written to", log_dir + "/train")

graph written to logs/fit/20260819-221455/train


#### Visualizing in Tensorboard

In [18]:
%load_ext tensorboard 
%tensorboard --logdir "logs/fit"